In [2]:
import zipfile
import xml.etree.ElementTree as ET
from io import BytesIO

In [5]:
def extract_reference_and_genotox(file_path):

    compound_name = None
    cas_number = None
    smiles = None

    with zipfile.ZipFile(file_path, 'r') as z:

        # =================================================
        # 1️⃣ REFERENCE_SUBSTANCE → IupacName / CASNumber
        # =================================================
        for fname in z.namelist():
            if not fname.endswith(".i6d"):
                continue

            with z.open(fname) as f:
                content = f.read()

            try:
                root = ET.parse(BytesIO(content)).getroot()
            except Exception:
                continue

            for elem in root.iter():
                tag = elem.tag.split("}")[-1]

                if tag == "REFERENCE_SUBSTANCE":
                    for sub in elem.iter():
                        sub_tag = sub.tag.split("}")[-1]
                        if sub_tag == "IupacName":
                            compound_name = sub.text
                        if sub_tag == "CASNumber":
                            cas_number = sub.text
                        if sub_tag == "SmilesNotation":   
                            smiles = sub.text
                    if compound_name or cas_number:
                        break

            if compound_name or cas_number:
                break

        print("\n==============================")
        print("COMPOUND IDENTIFICATION")
        print("==============================")
        print(f"Compound name (IUPAC): {compound_name}")
        print(f"CAS number           : {cas_number}")
        print(f"SMILES               : {smiles}")

        # =================================================
        # 2️⃣ 7.6.1 Genetic toxicity in vitro
        # =================================================
        print("\n==============================")
        print("7.6.1 GENETIC TOXICITY IN VITRO")
        print("==============================")

        def get_value(entry, field):
            for e in entry:
                if e.tag.split("}")[-1] == field:
                    for sub in e.iter():
                        if sub.tag.split("}")[-1] == "value":
                            return sub.text
            return None

        for fname in z.namelist():
            if not fname.endswith(".i6d"):
                continue

            with z.open(fname) as f:
                content = f.read()

            try:
                root = ET.parse(BytesIO(content)).getroot()
            except Exception:
                continue

            for study in root.iter():
                tag = study.tag.split("}")[-1]

                if tag != "ENDPOINT_STUDY_RECORD.GeneticToxicityVitro":
                    continue

                print(f"\n[FILE] {fname}")

                # ── MaterialsAndMethods ──────────────────────────
                for section in study:
                    if section.tag.split("}")[-1] != "MaterialsAndMethods":
                        continue

                    print("\n=== MaterialsAndMethods ===")

                    # Guideline entries
                    for child in section:
                        child_tag = child.tag.split("}")[-1]

                        if child_tag == "Guideline":
                            for idx, entry in enumerate(child, start=1):
                                qualifier  = get_value(entry, "Qualifier")
                                guideline  = get_value(entry, "Guideline")
                                deviation  = get_value(entry, "Deviation")
                                print(f"  Guideline #{idx}")
                                print(f"    Qualifier : {qualifier}")
                                print(f"    Guideline : {guideline}")
                                print(f"    Deviation : {deviation}")

                        elif child_tag == "GLPComplianceStatement":
                            for sub in child.iter():
                                if sub.tag.split("}")[-1] == "value":
                                    print(f"  GLP compliance        : {sub.text}")
                                    break

                        elif child_tag == "TypeOfAssay":
                            for sub in child.iter():
                                if sub.tag.split("}")[-1] == "value":
                                    print(f"  Type of assay         : {sub.text}")
                                    break

                        elif child_tag == "TestMaterials":
                            for sub in child.iter():
                                if sub.tag.split("}")[-1] == "TestMaterialInformation":
                                    print(f"  Test material info    : {sub.text}")
                                    break

                        elif child_tag == "Method":
                            for sub in child.iter():
                                sub_tag = sub.tag.split("}")[-1]
                                if sub_tag == "SpeciesStrain":
                                    # entry 아래 value
                                    for entry in sub:
                                        strain = get_value(entry, "SpeciesStrain")
                                        if strain:
                                            print(f"  Species/Strain        : {strain}")

                # ── ResultsAndDiscussion ─────────────────────────
                for section in study:
                    if section.tag.split("}")[-1] != "ResultsAndDiscussion":
                        continue

                    print("\n=== ResultsAndDiscussion ===")

                    for tr in section:
                        if tr.tag.split("}")[-1] != "TestRs":
                            continue

                        for idx, entry in enumerate(tr, start=1):
                            print(f"\n--- Test result #{idx} ---")
                            print(f"  Compound (IUPAC)               : {compound_name}")
                            print(f"  CAS number                     : {cas_number}")
                            print(f"  Key result                     : {get_value(entry, 'KeyResult')}")
                            print(f"  Species / strain               : {get_value(entry, 'Organism')}")
                            print(f"  Metabolic activation           : {get_value(entry, 'MetActIndicator')}")
                            print(f"  Genotoxicity                   : {get_value(entry, 'Genotoxicity')}")
                            print(f"  Cytotoxicity / top conc        : {get_value(entry, 'Cytotoxicity')}")
                            print(f"  Vehicle controls validity      : {get_value(entry, 'VehContrValid')}")
                            print(f"  Untreated negative ctrl valid  : {get_value(entry, 'NegContrValid')}")
                            print(f"  True negative ctrl valid       : None")
                            print(f"  Positive controls validity     : {get_value(entry, 'PosContrValid')}")

        print("\n==============================")
        print("END")
        print("==============================")


# ▶ 실행
extract_reference_and_genotox(
    r"dossier_test/compound.i6z"
)


COMPOUND IDENTIFICATION
Compound name (IUPAC): trichloro(phenyl)silane
CAS number           : 98-13-5
SMILES               : Cl[Si](Cl)(Cl)c1ccccc1

7.6.1 GENETIC TOXICITY IN VITRO

[FILE] 9913c595-132a-4005-a906-fc12dd22e856_000d6967-58f8-47eb-8f2e-a379174a5a56.i6d

=== MaterialsAndMethods ===
  Guideline #1
    Qualifier : 1680
    Guideline : 1289
    Deviation : 2158
  Guideline #2
    Qualifier : 1680
    Guideline : 1342
    Deviation : 2158
  GLP compliance        : 4179
  Type of assay         : 1049
  Test material info    : d6ccf045-fef9-4420-9058-aedbc530b108/000d6967-58f8-47eb-8f2e-a379174a5a56
  Species/Strain        : 3493

=== ResultsAndDiscussion ===

--- Test result #1 ---
  Compound (IUPAC)               : trichloro(phenyl)silane
  CAS number                     : 98-13-5
  Key result                     : None
  Species / strain               : 3493
  Metabolic activation           : 2474
  Genotoxicity                   : 2276
  Cytotoxicity / top conc        : 306

### 가이드라인이 두개이고 유전독성 결과가 positive인경우

In [6]:
import os
import zipfile
import xml.etree.ElementTree as ET
from io import BytesIO

def find_multi_guideline_with_genotox_2276(root_dir):

    results = []

    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            if not filename.endswith(".i6z"):
                continue

            file_path = os.path.join(dirpath, filename)

            try:
                z_outer = zipfile.ZipFile(file_path, 'r')
            except Exception:
                continue

            with z_outer:
                # i6z 안에 또 i6z가 있는 경우 처리
                inner_files = [n for n in z_outer.namelist() if n.endswith(".i6z")]
                i6d_files   = [n for n in z_outer.namelist() if n.endswith(".i6d")]

                def iter_i6d(z):
                    for fname in z.namelist():
                        if fname.endswith(".i6d"):
                            with z.open(fname) as f:
                                yield fname, f.read()

                def process_zip(z, source_label):
                    # REFERENCE_SUBSTANCE → compound_name, cas_number, smiles
                    compound_name = cas_number = smiles = None

                    for fname, content in iter_i6d(z):
                        try:
                            root = ET.parse(BytesIO(content)).getroot()
                        except Exception:
                            continue

                        for elem in root.iter():
                            if elem.tag.split("}")[-1] != "REFERENCE_SUBSTANCE":
                                continue
                            for sub in elem.iter():
                                t = sub.tag.split("}")[-1]
                                if t == "IupacName":    compound_name = sub.text
                                if t == "CASNumber":    cas_number    = sub.text
                                if t == "SmilesNotation": smiles      = sub.text
                            if compound_name or cas_number:
                                break
                        if compound_name or cas_number:
                            break

                    # GeneticToxicityVitro 순회
                    for fname, content in iter_i6d(z):
                        try:
                            root = ET.parse(BytesIO(content)).getroot()
                        except Exception:
                            continue

                        for study in root.iter():
                            if study.tag.split("}")[-1] != "ENDPOINT_STUDY_RECORD.GeneticToxicityVitro":
                                continue

                            # ── MaterialsAndMethods: 가이드라인 수집 ──
                            guidelines = []
                            for section in study:
                                if section.tag.split("}")[-1] != "MaterialsAndMethods":
                                    continue
                                for child in section:
                                    if child.tag.split("}")[-1] != "Guideline":
                                        continue
                                    for entry in child:
                                        def gv(node, field):
                                            for e in node:
                                                if e.tag.split("}")[-1] == field:
                                                    for s in e.iter():
                                                        if s.tag.split("}")[-1] == "value":
                                                            return s.text
                                            return None
                                        guidelines.append({
                                            "Qualifier": gv(entry, "Qualifier"),
                                            "Guideline": gv(entry, "Guideline"),
                                            "Deviation": gv(entry, "Deviation"),
                                        })

                            # 가이드라인 2개 이상인 경우만
                            if len(guidelines) < 2:
                                continue

                            # ── ResultsAndDiscussion: Genotoxicity == 2276 확인 ──
                            hit_entries = []
                            for section in study:
                                if section.tag.split("}")[-1] != "ResultsAndDiscussion":
                                    continue
                                for tr in section:
                                    if tr.tag.split("}")[-1] != "TestRs":
                                        continue
                                    for idx, entry in enumerate(tr, start=1):
                                        def gv2(node, field):
                                            for e in node:
                                                if e.tag.split("}")[-1] == field:
                                                    # KeyResult는 value 태그 없이 바로 text
                                                    if field == "KeyResult":
                                                        return e.text
                                                    for s in e.iter():
                                                        if s.tag.split("}")[-1] == "value":
                                                            return s.text
                                            return None

                                        genotox = gv2(entry, "Genotoxicity")
                                        if genotox == "2276":
                                            hit_entries.append({
                                                "result_index"  : idx,
                                                "KeyResult"     : gv2(entry, "KeyResult"),
                                                "Organism"      : gv2(entry, "Organism"),
                                                "MetActIndicator": gv2(entry, "MetActIndicator"),
                                                "Genotoxicity"  : genotox,
                                                "Cytotoxicity"  : gv2(entry, "Cytotoxicity"),
                                                "VehContrValid" : gv2(entry, "VehContrValid"),
                                                "NegContrValid" : gv2(entry, "NegContrValid"),
                                                "PosContrValid" : gv2(entry, "PosContrValid"),
                                            })

                            if hit_entries:
                                results.append({
                                    "source"       : source_label,
                                    "file"         : fname,
                                    "compound_name": compound_name,
                                    "cas_number"   : cas_number,
                                    "smiles"       : smiles,
                                    "guidelines"   : guidelines,
                                    "hit_entries"  : hit_entries,
                                })

                # 최상위 i6d 처리
                process_zip(z_outer, file_path)

                # 중첩 i6z 처리
                for inner_name in inner_files:
                    try:
                        with z_outer.open(inner_name) as inner_f:
                            inner_zip = zipfile.ZipFile(BytesIO(inner_f.read()), 'r')
                        with inner_zip:
                            process_zip(inner_zip, f"{file_path} > {inner_name}")
                    except Exception:
                        continue

    # ── 결과 출력 ──────────────────────────────────────────
    print(f"\n총 {len(results)}건 발견\n")
    print("=" * 70)

    for r in results:
        print(f"[SOURCE] {r['source']}")
        print(f"[FILE  ] {r['file']}")
        print(f"  Compound (IUPAC) : {r['compound_name']}")
        print(f"  CAS number       : {r['cas_number']}")
        print(f"  SMILES           : {r['smiles']}")
        print(f"  Guidelines ({len(r['guidelines'])}):")
        for i, g in enumerate(r['guidelines'], 1):
            print(f"    #{i}  Qualifier={g['Qualifier']}  Guideline={g['Guideline']}  Deviation={g['Deviation']}")
        print(f"  Genotoxicity=2276 hit results:")
        for h in r['hit_entries']:
            print(f"    --- Test result #{h['result_index']} ---")
            print(f"      KeyResult      : {h['KeyResult']}")
            print(f"      Organism       : {h['Organism']}")
            print(f"      MetActIndicator: {h['MetActIndicator']}")
            print(f"      Genotoxicity   : {h['Genotoxicity']}")
            print(f"      Cytotoxicity   : {h['Cytotoxicity']}")
            print(f"      VehContrValid  : {h['VehContrValid']}")
            print(f"      NegContrValid  : {h['NegContrValid']}")
            print(f"      PosContrValid  : {h['PosContrValid']}")
        print("=" * 70)

    return results


# ▶ 실행
results = find_multi_guideline_with_genotox_2276(
    r"data1/reach_study_results_dossiers_23-05-2023"
)


총 1334건 발견

[SOURCE] data1/reach_study_results_dossiers_23-05-2023/ac453128-294a-4d22-a531-1083e7fda60d.i6z
[FILE  ] d44332bd-289f-4c13-a353-ae89ad88a5c1_ac453128-294a-4d22-a531-1083e7fda60d.i6d
  Compound (IUPAC) : trihydrogen 17',18-dinitro-12λ³,12'λ³,14λ³,14'λ³-tetraoxa-1λ⁴,1'λ⁴,21,21'-tetraaza-13-chroma-13,13'-spirobi[pentacyclo[11.8.0.0²,¹¹.0³,⁸.0¹⁵,²⁰]henicosan]-1(21),1'(21'),2,2',4,4',6,6',8,8',10,10',15,15',17,17',19,19'-octadecaene-13,13,13-tris(ylium)-12,12',14,14'-tetraide 17,17'-dinitro-12λ³,12'λ³,14λ³,14'λ³-tetraoxa-1λ⁴,1'λ⁴,21,21'-tetraaza-13-chroma-13,13'-spirobi[pentacyclo[11.8.0.0²,¹¹.0³,⁸.0¹⁵,²⁰]henicosan]-1(21),1'(21'),2,2',4,4',6,6',8,8',10,10',15,15',17,17',19,19'-octadecaene-13,13,13-tris(ylium)-12,12',14,14'-tetraide 18,18'-dinitro-12λ³,12'λ³,14λ³,14'λ³-tetraoxa-1λ⁴,1'λ⁴,21,21'-tetraaza-13-chroma-13,13'-spirobi[pentacyclo[11.8.0.0²,¹¹.0³,⁸.0¹⁵,²⁰]henicosan]-1(21),1'(21'),2,2',4,4',6,6',8,8',10,10',15,15',17,17',19,19'-octadecaene-13,13,13-tris(ylium)-12,12',14,1

In [7]:
import os
import zipfile
import xml.etree.ElementTree as ET
from io import BytesIO
import csv

def find_multi_guideline_with_genotox_2276_to_csv(root_dir, output_csv="results.csv"):

    rows = []

    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            if not filename.endswith(".i6z"):
                continue

            file_path = os.path.join(dirpath, filename)
            source_filename = filename  # ac453128-...i6z 부분만

            try:
                z_outer = zipfile.ZipFile(file_path, 'r')
            except Exception:
                continue

            with z_outer:

                def iter_i6d(z):
                    for fname in z.namelist():
                        if fname.endswith(".i6d"):
                            with z.open(fname) as f:
                                yield fname, f.read()

                def get_ref(z):
                    compound_name = cas_number = smiles = None
                    for fname, content in iter_i6d(z):
                        try:
                            root = ET.parse(BytesIO(content)).getroot()
                        except Exception:
                            continue
                        for elem in root.iter():
                            if elem.tag.split("}")[-1] != "REFERENCE_SUBSTANCE":
                                continue
                            for sub in elem.iter():
                                t = sub.tag.split("}")[-1]
                                if t == "IupacName":      compound_name = sub.text
                                if t == "CASNumber":      cas_number    = sub.text
                                if t == "SmilesNotation": smiles        = sub.text
                            if compound_name or cas_number:
                                break
                        if compound_name or cas_number:
                            break
                    return compound_name, cas_number, smiles

                def gv(node, field):
                    for e in node:
                        if e.tag.split("}")[-1] == field:
                            if field == "KeyResult":
                                return e.text
                            for s in e.iter():
                                if s.tag.split("}")[-1] == "value":
                                    return s.text
                    return None

                def process_zip(z, src_fname):
                    compound_name, cas_number, smiles = get_ref(z)

                    for fname, content in iter_i6d(z):
                        try:
                            root = ET.parse(BytesIO(content)).getroot()
                        except Exception:
                            continue

                        for study in root.iter():
                            if study.tag.split("}")[-1] != "ENDPOINT_STUDY_RECORD.GeneticToxicityVitro":
                                continue

                            # 가이드라인 수집
                            guidelines = []
                            for section in study:
                                if section.tag.split("}")[-1] != "MaterialsAndMethods":
                                    continue
                                for child in section:
                                    if child.tag.split("}")[-1] != "Guideline":
                                        continue
                                    for entry in child:
                                        guidelines.append({
                                            "Qualifier": gv(entry, "Qualifier"),
                                            "Guideline": gv(entry, "Guideline"),
                                            "Deviation": gv(entry, "Deviation"),
                                        })

                            if len(guidelines) < 2:
                                continue

                            # Genotoxicity == 2276 확인
                            has_hit = False
                            hit_entries = []
                            for section in study:
                                if section.tag.split("}")[-1] != "ResultsAndDiscussion":
                                    continue
                                for tr in section:
                                    if tr.tag.split("}")[-1] != "TestRs":
                                        continue
                                    for idx, entry in enumerate(tr, start=1):
                                        if gv(entry, "Genotoxicity") == "2276":
                                            has_hit = True
                                            hit_entries.append({
                                                "result_index"   : idx,
                                                "KeyResult"      : gv(entry, "KeyResult"),
                                                "Organism"       : gv(entry, "Organism"),
                                                "MetActIndicator": gv(entry, "MetActIndicator"),
                                                "Genotoxicity"   : gv(entry, "Genotoxicity"),
                                                "Cytotoxicity"   : gv(entry, "Cytotoxicity"),
                                                "VehContrValid"  : gv(entry, "VehContrValid"),
                                                "NegContrValid"  : gv(entry, "NegContrValid"),
                                                "PosContrValid"  : gv(entry, "PosContrValid"),
                                            })

                            if not has_hit:
                                continue

                            # 가이드라인 수만큼 행 생성
                            for g in guidelines:
                                rows.append({
                                    "filename"       : src_fname,
                                    "compound_name"  : compound_name,
                                    "cas_number"     : cas_number,
                                    "smiles"         : smiles,
                                    "Qualifier"      : g["Qualifier"],
                                    "Guideline"      : g["Guideline"],
                                    "Deviation"      : g["Deviation"],
                                    "num_guidelines" : len(guidelines),
                                    "num_hits"       : len(hit_entries),
                                })

                # 최상위 처리
                process_zip(z_outer, source_filename)

                # 중첩 i6z 처리
                for inner_name in [n for n in z_outer.namelist() if n.endswith(".i6z")]:
                    try:
                        with z_outer.open(inner_name) as inner_f:
                            inner_zip = zipfile.ZipFile(BytesIO(inner_f.read()), 'r')
                        with inner_zip:
                            process_zip(inner_zip, source_filename)
                    except Exception:
                        continue

    # CSV 저장
    fieldnames = [
        "filename", "compound_name", "cas_number", "smiles",
        "Qualifier", "Guideline", "Deviation",
        "num_guidelines", "num_hits"
    ]

    with open(output_csv, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"저장 완료: {output_csv}  ({len(rows)}행)")
    return rows


# ▶ 실행
rows = find_multi_guideline_with_genotox_2276_to_csv(
    r"data1/reach_study_results_dossiers_23-05-2023",
    output_csv="genotox_positive_guidelines.csv"
)

저장 완료: genotox_positive_guidelines.csv  (3307행)
